# PCA-SVM Pipeline for Gait Pattern Classification

**Project:** SPM1d-PCA-SVM Framework for Quantifying Dynamic Gait Adaptation  
**Author:** Zhang Xining | Shanghai University of Sport  

This notebook implements the full machine learning pipeline:
1. Data loading from Visual 3D exports
2. Z-score standardization
3. PCA dimensionality reduction (≥95% variance)
4. Linear SVM classification with LOOCV
5. Feature back-projection to compute Contribution Scores
6. Discriminative phase identification (85th percentile threshold)

## 0. Dependencies

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False
print('Libraries loaded successfully.')

## 1. Configuration

Set `DATA_ROOT` to your data directory. Expected structure:
```
DATA_ROOT/
├── Habitual/
│   ├── Subject_01/
│   │   ├── *hip_joint*.xlsx
│   │   ├── *knee_joint*.xlsx
│   │   └── ...
│   └── Subject_02/ ...
└── Converted/
    ├── Subject_01/ ...
    └── ...
```

> **Note:** Raw motion capture data (`.xlsx` from Visual 3D) are not included due to participant privacy. The pipeline is fully functional with data following the Visual 3D export format.

In [ ]:
# ============================================================
# CONFIGURATION — set your data path here
# ============================================================
DATA_ROOT = r"path/to/your/data"  # <-- change this

# 21 kinematic channels: 7 segments × 3 planes
# Each entry: {'name': display_name, 'file_key': filename_substring, 'plane': 'X'/'Y'/'Z'}
FULL_CONFIG = [
    # Joint Angles
    {'name': 'Hip_Sagittal',      'file_key': 'hip_joint',    'plane': 'X'},
    {'name': 'Hip_Frontal',       'file_key': 'hip_joint',    'plane': 'Y'},
    {'name': 'Hip_Transverse',    'file_key': 'hip_joint',    'plane': 'Z'},
    {'name': 'Knee_Sagittal',     'file_key': 'knee_joint',   'plane': 'X'},
    {'name': 'Knee_Frontal',      'file_key': 'knee_joint',   'plane': 'Y'},
    {'name': 'Knee_Transverse',   'file_key': 'knee_joint',   'plane': 'Z'},
    {'name': 'Ankle_Sagittal',    'file_key': 'ankle_joint',  'plane': 'X'},
    {'name': 'Ankle_Frontal',     'file_key': 'ankle_joint',  'plane': 'Y'},
    {'name': 'Ankle_Transverse',  'file_key': 'ankle_joint',  'plane': 'Z'},
    # Segment Angles
    {'name': 'Pelvis_Sagittal',   'file_key': 'pelvis',       'plane': 'X'},
    {'name': 'Pelvis_Frontal',    'file_key': 'pelvis',       'plane': 'Y'},
    {'name': 'Pelvis_Transverse', 'file_key': 'pelvis',       'plane': 'Z'},
    {'name': 'Thigh_Sagittal',    'file_key': 'thigh',        'plane': 'X'},
    {'name': 'Thigh_Frontal',     'file_key': 'thigh',        'plane': 'Y'},
    {'name': 'Thigh_Transverse',  'file_key': 'thigh',        'plane': 'Z'},
    {'name': 'Shank_Sagittal',    'file_key': 'shank',        'plane': 'X'},
    {'name': 'Shank_Frontal',     'file_key': 'shank',        'plane': 'Y'},
    {'name': 'Shank_Transverse',  'file_key': 'shank',        'plane': 'Z'},
    {'name': 'Foot_Sagittal',     'file_key': 'foot',         'plane': 'X'},
    {'name': 'Foot_Frontal',      'file_key': 'foot',         'plane': 'Y'},
    {'name': 'Foot_Transverse',   'file_key': 'foot',         'plane': 'Z'},
]

print(f'Total kinematic channels: {len(FULL_CONFIG)}')
print(f'Total feature dimensions (whole-body): {len(FULL_CONFIG) * 101}')

## 2. Data Loading

In [ ]:
def load_curve(file_path, plane):
    """
    Load a single kinematic waveform from a Visual 3D Excel export.
    Returns a 101-point mean curve across trials, or None on failure.
    """
    try:
        df = pd.read_excel(file_path, header=4)
        df = df.iloc[:101, 1:]  # rows: stance time points; drop ITEM column
        col_offset = {'X': 0, 'Y': 1, 'Z': 2}[plane]
        plane_cols = df.iloc[:, col_offset::3].apply(pd.to_numeric, errors='coerce')
        mean_curve = plane_cols.mean(axis=1).values
        return mean_curve if len(mean_curve) == 101 else None
    except Exception:
        return None


def build_dataset(root_path, config):
    """
    Build the feature matrix X (n_subjects × n_features) and label vector y.
    n_features = n_channels × 101 time points = 21 × 101 = 2121 (whole-body)
    """
    groups = ['Habitual', 'Converted']
    X, y, names = [], [], []

    for group_idx, group in enumerate(groups):
        group_path = os.path.join(root_path, group)
        if not os.path.exists(group_path):
            print(f'[WARNING] Path not found: {group_path}')
            continue

        subjects = sorted([
            d for d in os.listdir(group_path)
            if os.path.isdir(os.path.join(group_path, d))
        ])

        for subj in subjects:
            subj_path = os.path.join(group_path, subj)
            curves, valid = [], True

            for ch in config:
                matched = [
                    f for f in os.listdir(subj_path)
                    if ch['file_key'] in f and f.endswith(('.xlsx', '.xls'))
                ]
                if not matched:
                    print(f'  [MISSING] {subj}: {ch["name"]}')
                    valid = False
                    break
                curve = load_curve(os.path.join(subj_path, matched[0]), ch['plane'])
                if curve is None:
                    valid = False
                    break
                curves.append(curve)

            if valid:
                X.append(np.concatenate(curves))
                y.append(group_idx)
                names.append(f'{group}_{subj}')

    return np.array(X), np.array(y), names


X, y, subject_names = build_dataset(DATA_ROOT, FULL_CONFIG)
print(f'Dataset: {X.shape[0]} subjects × {X.shape[1]} features')
print(f'Labels: {np.sum(y==0)} Habitual, {np.sum(y==1)} Converted')

## 3. PCA-SVM Classification with LOOCV

In [ ]:
# --- Step 1: Z-score standardization ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Step 2: PCA — retain components explaining ≥95% cumulative variance ---
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components = np.argmax(cumvar >= 0.95) + 1
print(f'PCA: {n_components} components explain {cumvar[n_components-1]*100:.2f}% variance')

X_svm = X_pca_full[:, :n_components]

# --- Step 3: Linear SVM with Leave-One-Out Cross-Validation ---
loo = LeaveOneOut()
svm = SVC(kernel='linear', C=1.0)

y_pred, y_true = [], []
svm_coefs = []  # store per-fold coefficients for back-projection

for train_idx, test_idx in loo.split(X_svm):
    svm.fit(X_svm[train_idx], y[train_idx])
    y_pred.append(svm.predict(X_svm[test_idx])[0])
    y_true.append(y[test_idx][0])
    svm_coefs.append(svm.coef_.copy())

accuracy = accuracy_score(y_true, y_pred)
print(f'\nWhole-body SVM Accuracy (LOOCV): {accuracy*100:.2f}%')
print(f'Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}')

## 4. Feature Back-Projection

Project SVM weights back through the PCA loading matrix to identify which original kinematic variables drive the classification.

**Formula:**
$$\text{Discriminative Weight}_{i,t} = \sum_{j=1}^{k} \left( W_{\text{svm},j} \times L_{\text{pca},j,i,t} \right)$$

**Contribution Score** for variable $i$ = $\sum_t |\text{Discriminative Weight}_{i,t}|$

In [ ]:
def feature_backprojection(pca_model, svm_coefs_list, n_comp, config, percentile_threshold=85):
    """
    Back-project LOOCV-averaged SVM weights through PCA loadings.
    Returns:
        contribution_scores: dict {variable_name: float}
        discriminative_weights: dict {variable_name: np.array(101)}
        discriminative_phases: dict {variable_name: list of time points}
    """
    # Average SVM coefficient across all LOOCV folds
    mean_coef = np.mean(np.vstack(svm_coefs_list), axis=0)  # shape: (1, n_comp)

    # PCA loadings: shape (n_components, n_original_features)
    L = pca_model.components_[:n_comp, :]

    # Reconstructed discriminative weight vector in original feature space
    # shape: (n_original_features,) = (21 * 101,)
    w_original = mean_coef @ L  # (1, n_comp) @ (n_comp, n_features) = (1, n_features)
    w_original = w_original.flatten()

    # Split back into per-variable weight curves
    contribution_scores = {}
    discriminative_weights = {}
    discriminative_phases = {}

    T = 101
    for i, ch in enumerate(config):
        name = ch['name']
        w_var = w_original[i*T : (i+1)*T]  # shape: (101,)

        contribution_scores[name] = float(np.sum(np.abs(w_var)))
        discriminative_weights[name] = w_var

        # Discriminative phase: time points where |weight| > 85th percentile
        threshold = np.percentile(np.abs(w_original), percentile_threshold)
        phases = np.where(np.abs(w_var) > threshold)[0].tolist()
        discriminative_phases[name] = phases

    return contribution_scores, discriminative_weights, discriminative_phases


scores, weights, phases = feature_backprojection(
    pca_full, svm_coefs, n_components, FULL_CONFIG
)

# Sort and display Contribution Scores
df_scores = pd.DataFrame(
    sorted(scores.items(), key=lambda x: x[1], reverse=True),
    columns=['Variable', 'Contribution Score']
)
df_scores.index = df_scores.index + 1
print('\nContribution Score Ranking (Top 10):')
print(df_scores.head(10).to_string())

## 5. Visualization

In [ ]:
# --- Plot 1: Contribution Score bar chart ---
fig, ax = plt.subplots(figsize=(10, 7), dpi=150)

top_n = 10
df_top = df_scores.head(top_n)
colors = ['#D62728' if 'Knee_Sagittal' in v else '#1F77B4' for v in df_top['Variable']]

bars = ax.barh(df_top['Variable'][::-1], df_top['Contribution Score'][::-1],
               color=colors[::-1], edgecolor='white', height=0.7)

ax.set_xlabel('Contribution Score (Sum of Absolute Back-projected Weights)', fontsize=11)
ax.set_title('Whole-body SVM: Feature Contribution to Gait Classification\n'
             '(Red = SPM1d non-significant but SVM top contributor)', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('contribution_scores.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: contribution_scores.png')

In [ ]:
# --- Plot 2: Discriminative weight curve for top variable (Knee Sagittal) ---
top_var = df_scores.iloc[0]['Variable']
w = weights[top_var]
disc_phase = phases[top_var]

time = np.arange(101)
fig, ax = plt.subplots(figsize=(9, 4), dpi=150)

ax.plot(time, w, color='#2C7BB6', linewidth=2, label='Discriminative Weight')
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')

# Shade discriminative phase
if disc_phase:
    ax.axvspan(min(disc_phase), max(disc_phase), alpha=0.2, color='#D62728',
               label=f'Discriminative Phase ({min(disc_phase)}%–{max(disc_phase)}% stance)')

ax.set_xlabel('Stance Phase (%)', fontsize=11)
ax.set_ylabel('Discriminative Weight', fontsize=11)
ax.set_title(f'Feature Back-projection: {top_var}\n(Top contributor to whole-body SVM)', fontsize=12)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'discriminative_weight_{top_var}.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: discriminative_weight_{top_var}.png')

## 6. Summary

| Metric | Value |
|--------|-------|
| Subjects | 24 (12 HAB + 12 RET) |
| Feature dimensions (whole-body) | 2121 (21 vars × 101 time points) |
| PCA components retained | see output above |
| Whole-body SVM accuracy (LOOCV) | **71.05%** |
| Top discriminative feature | **Knee Sagittal (X)** — non-significant in SPM1d |
| Discriminative phase | **16–30% stance** (mid-stance) |

**Key insight:** The SVM identified Knee Sagittal Angle as the #1 discriminator despite it being statistically non-significant in SPM1d — demonstrating that *statistical significance ≠ motion discriminability*.